# 0. Environment Setup

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## 0.1 Clone Repository & Install Dependencies

In [2]:
REPO_URL = "https://github.com/11erlangga/legal-rag-slm.git"
REPO_DIR = "/kaggle/working/repo"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 98 (delta 56), reused 75 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 404.40 KiB | 2.02 MiB/s, done.
Resolving deltas: 100% (56/56), done.


In [3]:
!pip install -q "transformers==4.57.3" "accelerate==1.12.0" "bitsandbytes==0.49.1"
!pip install -q "langchain==1.2.3" "langchain-community==0.4.1" "langchain-core==1.2.6"
!pip install -q langchain-classic langchain-text-splitters langchain-huggingface langchain-chroma
!pip install -q "chromadb==1.4.1" "huggingface-hub==0.36.0"
!pip install -q rank-bm25 sentence-transformers pymupdf ddgs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 798.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 489.1/489.1 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curre

## 0.2 Import Libraries & Secrets

In [4]:
import sys
sys.path.append(REPO_DIR)

import torch
from typing import Dict, Optional

from src.rag.pipeline import (
    build_retrievers, build_generator, build_pipeline,
    RAGPipeline, sanity_check_retrieval,
)
from src.rag.hyde import build_hyde_pipeline, generate_hypothetical_answers, hyde_retrieve
from src.rag.retrievers import print_retrieved_docs, rerank_with_scores
from src.rag.web_fallback import retrieve_with_fallback

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

In [5]:
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU Detected: {gpu_stats.name}. Max Memory = {max_memory} GB.")
else:
    print("WARNING: GPU Not Detected.")

GPU Detected: Tesla T4. Max Memory = 14.562 GB.


# 1. Config

- `PDF_DIR`: path ke Kaggle Dataset berisi 4 PDF UU.
- `HF_REPO_ID`: repo hasil fine-tuning.

In [6]:
PDF_DIR = "/kaggle/input/datasets/erlanggasriheryanto/uu-legal-docs"
HF_REPO_ID = "11erlangga/grpo-qwen25-3b"
SANITY_QUERY = "Apa saja hak pekerja lembur menurut PP 35/2021?"
ENSEMBLE_WEIGHTS = (0.75, 0.25) # BM25, semantic

# 2. Build & Compare Retriever Modes

In [7]:
retrievers = build_retrievers(PDF_DIR, ensemble_weights=ENSEMBLE_WEIGHTS)

Parent chunk size: 2000, overlap: 200
Child chunk size: 400, overlap: 50


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [8]:
sanity_check_retrieval(retrievers["dense"], SANITY_QUERY)

Query: Apa saja hak pekerja lembur menurut PP 35/2021?

--- Dokumen 1 ---
Sumber: UU Nomor 6 Tahun 2023.pdf, halaman: 558
UU/PP: UU No. 6 Tahun 2023, Pasal terdeteksi: 79
PRESIDEN
REPI.IBLIK INDONESIA
-549-
(21 Pengusaha yang mempekerjakan Pekerja/Buruh
melebihi waktu kerja sebagaimana dimaksud pada
ayat (1) wajib membayar Upah kerja lembur.
(3) Ketentuan waktu kerja lembur sebagaimana
dimaksud pada ayat (1) hurrrf b tidak berlaku bagi
sektor usaha atau pekerjaan tertentu.
(4) Ketentuan lebih lanjut mengenai waktu kerja
lembur dan Upah kerja lembur diatur dalam
Peraturan Pemerintah.
25. Ketentuan Pasal 79 diubah sehingga berbunyi sebagai
berikut:
Pasal 79
(1) Pengusaha wajib memberi:
a- waktu istirahat; dan
b. cuti.
(21 Waktu istirahat sebagaimana dimaksud pada ayat
(1) huruf a wajib diberikan kepada Pekerja/Buruh
paling sedikit meliputi:
a. istirahat antara jam kerja, paling sedikit
setengah jam setelah bekerja selama 4 (empat)
jam terus-menerus, dan waktu istirahat
tersebut tidak ter

**Observasi dense-only:** _(isi setelah lihat hasil retrieval di atas -- chunk apa yang muncul, relevan atau tidak, dst.)_

In [9]:
sanity_check_retrieval(retrievers["hybrid"], SANITY_QUERY)

Query: Apa saja hak pekerja lembur menurut PP 35/2021?

--- Dokumen 1 ---
Sumber: PP Nomor 5 Tahun 2021.pdf, halaman: 334
UU/PP: PP No. 5 Tahun 2021, Pasal terdeteksi: (tidak ada)
atas 
persetujuan 
pihak 
keluarga 
pekerja 
migran 
Indonesia atau sesuai dengan ketentuan yang 
berlaku di negara yang bersangkutan; 
m. 
tidak memberikan perlindungan terhadap seluruh 
harta milik pekerja migran Indonesia untuk 
kepentingan keluarganya; 
n. 
tidak mengurus pemenuhan semua hak pekerja 
migran Indonesia yang seharusnya diterima; 
o. 
tidak memulangkan pekerja migran Indonesia

--- Dokumen 2 ---
Sumber: PP Nomor 35 Tahun 2021.pdf, halaman: 17
UU/PP: PP No. 35 Tahun 2021, Pasal terdeteksi: 23, 2l, 30, 31
PRESiDEN
REPUBLIK INDONESI,A.
-18-
(21 Perintah dan persetujuan sebagaimana dimaksud
pada ayat (1) dapat dibuat dalam bentuk daftar
Pekerja/Buruh yang bersedia bekerja lembur yang
ditandatangani oleh 
Pekerja/Buruh yang
bersangkutan dan Pengusaha.
(3) Pengusaha sebagaimana dimaksud pada ayat(2

**Observasi hybrid (BM25+dense):** _(bandingkan sama dense-only di atas -- ada chunk baru yang lebih relevan? urutan berubah?)_

In [10]:
sanity_check_retrieval(retrievers["hybrid_rerank"], SANITY_QUERY)

Query: Apa saja hak pekerja lembur menurut PP 35/2021?

--- Dokumen 1 ---
Sumber: UU Nomor 6 Tahun 2023.pdf, halaman: 558
UU/PP: UU No. 6 Tahun 2023, Pasal terdeteksi: 79
PRESIDEN
REPI.IBLIK INDONESIA
-549-
(21 Pengusaha yang mempekerjakan Pekerja/Buruh
melebihi waktu kerja sebagaimana dimaksud pada
ayat (1) wajib membayar Upah kerja lembur.
(3) Ketentuan waktu kerja lembur sebagaimana
dimaksud pada ayat (1) hurrrf b tidak berlaku bagi
sektor usaha atau pekerjaan tertentu.
(4) Ketentuan lebih lanjut mengenai waktu kerja
lembur dan Upah kerja lembur diatur dalam
Peraturan Pemerintah.
25. Ketentuan Pasal 79 diubah sehingga berbunyi sebagai
berikut:
Pasal 79
(1) Pengusaha wajib memberi:
a- waktu istirahat; dan
b. cuti.
(21 Waktu istirahat sebagaimana dimaksud pada ayat
(1) huruf a wajib diberikan kepada Pekerja/Buruh
paling sedikit meliputi:
a. istirahat antara jam kerja, paling sedikit
setengah jam setelah bekerja selama 4 (empat)
jam terus-menerus, dan waktu istirahat
tersebut tidak ter

**Observasi hybrid+reranker:** _(apakah top-3 hasil reranker memang lebih relevan dibanding hybrid tanpa reranker? worth latency tambahannya?)_

# 3. HyDE

In [11]:
llm, tokenizer = build_generator(HF_REPO_ID, hf_token=HF_TOKEN)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

Device set to use cuda:0


In [12]:
hyde_llm = build_hyde_pipeline(llm)

Device set to use cuda:0


In [13]:
hyde_query = SANITY_QUERY

hyde_answers = generate_hypothetical_answers(
    hyde_query, hyde_llm, tokenizer, n=2
)

print(f"Query asli: {hyde_query}\n")
for i, ans in enumerate(hyde_answers, start=1):
    print(f"[Hipotesis {i}]")
    print(ans)
    print()

Query asli: Apa saja hak pekerja lembur menurut PP 35/2021?

[Hipotesis 1]
Dokumen ini mencakup informasi tentang hak-hak pekerja lembur menurut Peraturan Pemerintah Nomor 35 Tahun 2021.

[Hipotesis 2]
Hak pekerja lembur sesuai dengan Undang-Undang No. 17 tahun 2018 tentang Perlindungan Ketenagakerjaan di Indonesia, terutama dalam Pasal 34 yang mengajukan waktu kerja maksimal sehari sebesar 8 jam dan waktu kerja total maksimum dalam seminggu sebesar 48 jam. Selain itu, ada hak-hak lain seperti:

1) Hak cuti bersama: Pekerja diberikan cuti yang dapat digunakan untuk mengambil liburan atau masa liburan khusus.
2) Hak pensiun: Pekerja diperlakukan dengan adil dan layak di usia pensiun.
3) Hak jaminan sosial: Pekerja mendapatkan akses ke program jaminan sosial, termasuk BPJS dan jaminan hari tua.

Pada akhirnya, hak pekerja



In [14]:
hyde_docs = hyde_retrieve(hyde_query, hyde_answers, retrievers["hybrid"], top_k=10)

print(f"Total dokumen unik hasil union HyDE: {len(hyde_docs)}\n")
print_retrieved_docs(hyde_docs)

Total dokumen unik hasil union HyDE: 10

--- Dokumen 1 ---
Sumber: PP Nomor 5 Tahun 2021.pdf, halaman: 334
UU/PP: PP No. 5 Tahun 2021, Pasal terdeteksi: (tidak ada)
atas 
persetujuan 
pihak 
keluarga 
pekerja 
migran 
Indonesia atau sesuai dengan ketentuan yang 
berlaku di negara yang bersangkutan; 
m. 
tidak memberikan perlindungan terhadap seluruh 
harta milik pekerja migran Indonesia untuk 
kepentingan keluarganya; 
n. 
tidak mengurus pemenuhan semua hak pekerja 
migran Indonesia yang seharusnya diterima; 
o. 
tidak memulangkan pekerja migran Indonesia

--- Dokumen 2 ---
Sumber: PP Nomor 35 Tahun 2021.pdf, halaman: 17
UU/PP: PP No. 35 Tahun 2021, Pasal terdeteksi: 23, 2l, 30, 31
PRESiDEN
REPUBLIK INDONESI,A.
-18-
(21 Perintah dan persetujuan sebagaimana dimaksud
pada ayat (1) dapat dibuat dalam bentuk daftar
Pekerja/Buruh yang bersedia bekerja lembur yang
ditandatangani oleh 
Pekerja/Buruh yang
bersangkutan dan Pengusaha.
(3) Pengusaha sebagaimana dimaksud pada ayat(2) harus
membuat

In [15]:
baseline_docs = retrievers["hybrid"].invoke(hyde_query)
print(f"Total dokumen hasil retrieval TANPA HyDE: {len(baseline_docs)}\n")
print_retrieved_docs(baseline_docs)

Total dokumen hasil retrieval TANPA HyDE: 16

--- Dokumen 1 ---
Sumber: PP Nomor 5 Tahun 2021.pdf, halaman: 334
UU/PP: PP No. 5 Tahun 2021, Pasal terdeteksi: (tidak ada)
atas 
persetujuan 
pihak 
keluarga 
pekerja 
migran 
Indonesia atau sesuai dengan ketentuan yang 
berlaku di negara yang bersangkutan; 
m. 
tidak memberikan perlindungan terhadap seluruh 
harta milik pekerja migran Indonesia untuk 
kepentingan keluarganya; 
n. 
tidak mengurus pemenuhan semua hak pekerja 
migran Indonesia yang seharusnya diterima; 
o. 
tidak memulangkan pekerja migran Indonesia

--- Dokumen 2 ---
Sumber: PP Nomor 35 Tahun 2021.pdf, halaman: 17
UU/PP: PP No. 35 Tahun 2021, Pasal terdeteksi: 23, 2l, 30, 31
PRESiDEN
REPUBLIK INDONESI,A.
-18-
(21 Perintah dan persetujuan sebagaimana dimaksud
pada ayat (1) dapat dibuat dalam bentuk daftar
Pekerja/Buruh yang bersedia bekerja lembur yang
ditandatangani oleh 
Pekerja/Buruh yang
bersangkutan dan Pengusaha.
(3) Pengusaha sebagaimana dimaksud pada ayat(2) harus
me

# 4. Reranker Relevance Score Extraction + Threshold + Fallback DuckDuckGo

> Ekstrak skor Top-1 hasil reranker -> kalau di bawah threshold, abaikan dokumen lokal dan fallback ke DuckDuckGo Search.

In [16]:
FALLBACK_THRESHOLD = 0.0  # BELUM dikalibrasi empiris

In [17]:
# Sanity check skor
check_queries = {
    "in_domain": SANITY_QUERY,
    "out_of_domain": "Bagaimana prosedur perceraian di pengadilan agama?",
}

for label, q in check_queries.items():
    candidates = retrievers["hybrid"].invoke(q)
    ranked = rerank_with_scores(retrievers["reranker"], q, candidates)
    top_score = ranked[0][1] if ranked else None
    print(f"[{label}] query: {q!r}")
    print(f"  -> top_score: {top_score}\n")

[in_domain] query: 'Apa saja hak pekerja lembur menurut PP 35/2021?'
  -> top_score: 0.9987624883651733

[out_of_domain] query: 'Bagaimana prosedur perceraian di pengadilan agama?'
  -> top_score: 0.02578010782599449



In [18]:
# Demonstrasi if-else threshold: satu query lokal, satu query fallback
demo_queries = {
    "expect_local": SANITY_QUERY,
    "expect_fallback": "Apa hukuman untuk pelanggaran lalu lintas di jalan tol?",
}

for label, q in demo_queries.items():
    result = retrieve_with_fallback(
        q, retrievers["hybrid"], retrievers["reranker"],
        threshold=FALLBACK_THRESHOLD, top_n=3,
    )
    print(f"[{label}] query: {q!r}")
    print(f"  used_fallback: {result['used_fallback']}")
    print(f"  top_score: {result['top_score']}\n")
    print_retrieved_docs(result["docs"])

[expect_local] query: 'Apa saja hak pekerja lembur menurut PP 35/2021?'
  used_fallback: False
  top_score: 0.9987624883651733

--- Dokumen 1 ---
Sumber: UU Nomor 6 Tahun 2023.pdf, halaman: 558
UU/PP: UU No. 6 Tahun 2023, Pasal terdeteksi: 79
PRESIDEN
REPI.IBLIK INDONESIA
-549-
(21 Pengusaha yang mempekerjakan Pekerja/Buruh
melebihi waktu kerja sebagaimana dimaksud pada
ayat (1) wajib membayar Upah kerja lembur.
(3) Ketentuan waktu kerja lembur sebagaimana
dimaksud pada ayat (1) hurrrf b tidak berlaku bagi
sektor usaha atau pekerjaan tertentu.
(4) Ketentuan lebih lanjut mengenai waktu kerja
lembur dan Upah kerja lembur diatur dalam
Peraturan Pemerintah.
25. Ketentuan Pasal 79 diubah sehingga berbunyi sebagai
berikut:
Pasal 79
(1) Pengusaha wajib memberi:
a- waktu istirahat; dan
b. cuti.
(21 Waktu istirahat sebagaimana dimaksud pada ayat
(1) huruf a wajib diberikan kepada Pekerja/Buruh
paling sedikit meliputi:
a. istirahat antara jam kerja, paling sedikit
setengah jam setelah bekerja se